# Day 31：Week 5 周测与 RAG 调优实验台

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第三阶段 Day 31！今天我们将实战 `TuningConfig` 与 `run_grid_experiment`，通过网格搜索遍历不同的 `chunk_size`、`chunk_overlap` 与 `top_k`，用 **Hit Rate@K** 和 **MRR** 两个核心指标科学锁定最优 RAG 检索配置（`v0.3.0-day31`）。

## 1. 环境初始化与导入 `project_iterations/day_31` 模块
定位仓库根目录并将 `project_iterations/day_31` 加入 `sys.path`，导入 `TuningConfig`、`EvalQAPair`、`chunk_text` 与 `run_grid_experiment`。

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day31_dir = repo_root / "project_iterations" / "day_31"
if str(day31_dir) not in sys.path:
    sys.path.insert(0, str(day31_dir))

from zhishu_agent import EvalQAPair, TuningConfig, chunk_text, run_grid_experiment

print("已加载模块路径:", day31_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_31


## 2. 文本切分与重叠窗口演示（`chunk_text`）
对比无重叠与带重叠窗口的切分效果，观察重叠区如何保护跨边界语义不被割裂。

In [2]:
sample_policy = "星辰数科员工休假制度：入职满1年的正式员工每年享有5天带薪年假，入职满10年享有10天带薪年假。"
chunks_no_overlap = chunk_text(sample_policy, chunk_size=24, chunk_overlap=0)
chunks_with_overlap = chunk_text(sample_policy, chunk_size=24, chunk_overlap=8)

print("无重叠切分结果:", chunks_no_overlap)
print("带重叠(overlap=8)切分结果:", chunks_with_overlap)

无重叠切分结果: ['星辰数科员工休假制度：入职满1年的正式员工每年享', '有5天带薪年假，入职满10年享有10天带薪年假。']
带重叠(overlap=8)切分结果: ['星辰数科员工休假制度：入职满1年的正式员工每年享', '的正式员工每年享有5天带薪年假，入职满10年享有', '入职满10年享有10天带薪年假。']


## 3. 运行 RAG 超参数网格实验（`run_grid_experiment`）
构建星辰数科企业知识库语料与 4 条标准评测问答对，遍历 4 组 `TuningConfig` 参数组合，输出 Hit Rate@K 与 MRR 排行榜。

In [3]:
corpus = [
    {
        "id": "hr-policy",
        "content": "星辰数科员工休假制度：入职满1年的正式员工每年享有5天带薪年假，入职满10年享有10天带薪年假。请假超过3天需由部门总监审批。",
    },
    {
        "id": "fin-reimburse",
        "content": "差旅报销规范：一线城市酒店住宿标准上限为每晚600元，二线城市为每晚450元。员工出差需在返程后7个工作日内通过智枢财务系统提交电子发票。",
    },
    {
        "id": "it-redis",
        "content": "生产环境 Redis 运维红线：严禁在生产主库执行 KEYS * 或 FLUSHALL 命令，批量遍历键空间必须使用 SCAN 命令并控制 COUNT 参数。",
    },
    {
        "id": "rag-arch",
        "content": "智枢 RAG 检索架构：采用递归切分与向量检索结合，支持按 department 元数据预过滤，默认推荐使用 chunk_size=80、chunk_overlap=20 与 top_k=2。",
    },
]

eval_qa_pairs = [
    EvalQAPair(query="入职满1年有几天带薪年假？", expected_keywords=("5天带薪年假",), relevant_doc_ids=("hr-policy",)),
    EvalQAPair(query="一线城市出差住宿每晚报销上限是多少？", expected_keywords=("600元",), relevant_doc_ids=("fin-reimburse",)),
    EvalQAPair(query="生产环境 Redis 禁止执行什么命令？", expected_keywords=("KEYS *", "SCAN"), relevant_doc_ids=("it-redis",)),
    EvalQAPair(query="智枢 RAG 默认推荐什么切分和检索参数？", expected_keywords=("chunk_size=80", "top_k=2"), relevant_doc_ids=("rag-arch",)),
]

configs = [
    TuningConfig(chunk_size=30, chunk_overlap=0, top_k=1),
    TuningConfig(chunk_size=50, chunk_overlap=10, top_k=2),
    TuningConfig(chunk_size=80, chunk_overlap=20, top_k=2),
    TuningConfig(chunk_size=120, chunk_overlap=20, top_k=3),
]

report = run_grid_experiment(corpus=corpus, eval_qa_pairs=eval_qa_pairs, configs=configs)
print(report.summary_table())
print(f"\n最优配置: {report.best_config.label} | Hit Rate@K={report.best_result.hit_rate:.2%}, MRR={report.best_result.mrr:.4f}")

Rank  | chunk_size | overlap  | top_k  | Chunks  | Hit Rate@K  | MRR     
-------------------------------------------------------------------------
1     | 120        | 20       | 3      | 4       | 1.0000      | 1.0000  
2     | 80         | 20       | 2      | 5       | 0.7500      | 0.7500  
3     | 50         | 10       | 2      | 9       | 0.5000      | 0.5000  
4     | 30         | 0        | 1      | 13      | 0.2500      | 0.2500  

最优配置: chunk=120, overlap=20, top_k=3 | Hit Rate@K=100.00%, MRR=1.0000


## 4. 逐题检索命中诊断与断言校验
查看最优配置下每条 Query 的首次命中排名（`first_relevant_rank`）与倒数排名（`reciprocal_rank`），并校验实验结果。

In [4]:
for item in report.best_result.per_query_details:
    print(f"Query: {item['query']} | hit={item['hit']} | rank={item['first_relevant_rank']} | RR={item['reciprocal_rank']:.2f}")

assert report.best_result.hit_rate == 1.0
assert report.best_result.mrr == 1.0
print("Day 31 RAG 网格调优实验校验通过！")

Query: 入职满1年有几天带薪年假？ | hit=True | rank=1 | RR=1.00
Query: 一线城市出差住宿每晚报销上限是多少？ | hit=True | rank=1 | RR=1.00
Query: 生产环境 Redis 禁止执行什么命令？ | hit=True | rank=1 | RR=1.00
Query: 智枢 RAG 默认推荐什么切分和检索参数？ | hit=True | rank=1 | RR=1.00
Day 31 RAG 网格调优实验校验通过！
